# Phase 8 — Causal Interventions

**Goal:** Verify that the representations identified through probing and SAE analysis **causally** affect ranking behavior — not just correlate with it.

Two experiment families:
1. **Probe-direction steering** — inject `±α·w` into residual stream at decision token
2. **SAE feature steering** — zero-ablate or amplify individual SAE features at layer 17

**Key question (probe faithfulness):** Does the probe strength (R²/AUROC from Phase 6) predict causal effect size (ΔnDCG under steering)?

---
Run experiments first:
```bash
# Recommended: probe-only first (faster), then SAE
python -m src.interventions.runner --dataset scifact --probe_only
python -m src.interventions.runner --dataset scifact --sae_only
# Or both together:
python -m src.interventions.runner --dataset scifact
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr

sns.set_theme(style="whitegrid", font_scale=1.15)

DATASET = "scifact"
RESULTS_DIR = Path("../outputs/final/interventions") / DATASET
FIGURES_DIR = Path("../outputs/final/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Results

In [ ]:
# Load baseline
with open(RESULTS_DIR / "baseline_metrics.json") as f:
    baseline_metrics = json.load(f)
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Load probe intervention results
with open(RESULTS_DIR / "probe_intervention_results.json") as f:
    probe_raw = json.load(f)

probe_df = pd.DataFrame(probe_raw)
print(f"Probe conditions: {len(probe_df)}")
print(probe_df[["target", "layer", "alpha_multiplier", "delta_ndcg", "delta_mrr", "p_value", "significant"]].head(12))

In [ ]:
# Load SAE results
with open(RESULTS_DIR / "sae_intervention_results.json") as f:
    sae_raw = json.load(f)

sae_df = pd.DataFrame(sae_raw)
print(f"SAE conditions: {len(sae_df)}")

# Feature plan: which feature was selected for each IR target and its correlation sign
feature_plan = (
    sae_df[["feature_idx", "ir_target", "r_value"]]
    .drop_duplicates("feature_idx")
    .sort_values("ir_target")
)
print("\nFeature plan (top feature per IR target):")
print(feature_plan.to_string(index=False))
print("\nNote: r_value < 0 → feature fires for LOW-target-value docs.")
print("  Ablating a negative-r feature removes a suppressive signal.")
print("  Amplifying a negative-r feature pushes toward lower target values.")

## 2. Probe Steering — Dose-Response Curves

For each target, how does ΔnDCG@10 change as α sweeps from −5 to +5?

In [ ]:
# Focus on layer 17 (confirmed peak)
layer17 = probe_df[probe_df["layer"] == 17].copy()

targets = layer17["target"].unique()
fig, axes = plt.subplots(1, len(targets), figsize=(5 * len(targets), 4), sharey=True)
if len(targets) == 1:
    axes = [axes]

colors = sns.color_palette("tab10", len(targets))

for ax, target, color in zip(axes, targets, colors):
    sub = layer17[layer17["target"] == target].sort_values("alpha_multiplier")
    ax.plot(sub["alpha_multiplier"], sub["delta_ndcg"], marker="o", color=color, label=target)
    
    # Highlight significant points
    sig = sub[sub["significant"]]
    ax.scatter(sig["alpha_multiplier"], sig["delta_ndcg"], s=120, marker="*",
               color="red", zorder=5, label="p<0.05")
    
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(target.replace("_", " ").title())
    ax.set_xlabel("α multiplier")
    ax.legend(fontsize=9)

axes[0].set_ylabel("ΔnDCG@10")
fig.suptitle(f"Probe-Direction Steering — Layer 17 ({DATASET})", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "intervention_probe_dose_response.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Probe Steering — Layer Comparison

Does layer 17 show stronger causal effects than layers 7 or 21?

In [ ]:
# Best (max |ΔnDCG|) per (target, layer) regardless of alpha sign
best_by_layer = (
    probe_df.groupby(["target", "layer"])
    .apply(lambda g: g.loc[g["delta_ndcg"].abs().idxmax()])
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(8, 4))
pivot = best_by_layer.pivot(index="layer", columns="target", values="delta_ndcg")
pivot.plot(kind="bar", ax=ax, colormap="tab10", edgecolor="black", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Layer")
ax.set_ylabel("ΔnDCG@10 (best α)")
ax.set_title(f"Best Causal Effect per Layer — Probe Steering ({DATASET})", fontweight="bold")
ax.legend(title="Target", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "intervention_probe_layer_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Probe Faithfulness Analysis

Do features with higher probe strength (R²/AUROC) produce larger causal effects?

This directly answers the core question: **when does a linear probe imply causal use?**

In [ ]:
# Load probe results (R²/AUROC scores per target)
with open(f"../data/processed/{DATASET}/probe_results.json") as f:
    probe_results_raw = json.load(f)

probe_results_df = pd.DataFrame(probe_results_raw)

# Peak probe score per target at layer 17
peak_probe = (
    probe_results_df[probe_results_df["layer"] == 17]
    .set_index("target")[["score"]]
    .rename(columns={"score": "probe_score_layer17"})
)

# Best causal effect per target at layer 17
best_causal = (
    probe_df[probe_df["layer"] == 17]
    .groupby("target")["delta_ndcg"]
    .apply(lambda x: x.abs().max())
    .rename("best_delta_ndcg")
)

faithfulness = peak_probe.join(best_causal, how="inner").dropna()

if len(faithfulness) >= 3:
    r, p = pearsonr(faithfulness["probe_score_layer17"], faithfulness["best_delta_ndcg"])
    print(f"Probe faithfulness correlation: r={r:.3f}, p={p:.3f}")
    print(faithfulness)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(faithfulness["probe_score_layer17"], faithfulness["best_delta_ndcg"],
               s=120, color="steelblue", zorder=3)
    for target, row in faithfulness.iterrows():
        ax.annotate(target.replace("_", "\n"), (row["probe_score_layer17"], row["best_delta_ndcg"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=9)
    ax.set_xlabel("Probe Score (R² or AUROC at Layer 17)")
    ax.set_ylabel("|ΔnDCG@10| (best steering)")
    ax.set_title(f"Probe Faithfulness — {DATASET}\nr={r:.3f}, p={p:.3f}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "intervention_probe_faithfulness.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough data points for faithfulness analysis")
    print(faithfulness)

## 5. SAE Feature Steering Results

In [ ]:
# SAE steering full results — grouped by IR target
print("SAE Feature Steering — Full Results")
display_cols = ["ir_target", "feature_idx", "r_value", "mode", "alpha",
                "delta_ndcg", "delta_mrr", "p_value", "significant", "collapsed"]
print(sae_df[display_cols].sort_values(["ir_target", "mode", "alpha"]).to_string(index=False))

In [ ]:
# SAE: best |ΔnDCG| per (ir_target, mode) — one bar per target/mode combination
best_sae = (
    sae_df.groupby(["ir_target", "mode"])
    .apply(lambda g: g.loc[g["delta_ndcg"].abs().idxmax()])
    .reset_index(drop=True)
)

ir_targets = sorted(best_sae["ir_target"].unique())
n = len(ir_targets)
palette = {"ablate": "#e74c3c", "amplify": "#2ecc71"}

fig, ax = plt.subplots(figsize=(max(10, n * 1.8), 4))
x = np.arange(n)
width = 0.35

for i, mode in enumerate(["ablate", "amplify"]):
    sub = best_sae[best_sae["mode"] == mode].set_index("ir_target").reindex(ir_targets)
    vals = sub["delta_ndcg"].fillna(0).values
    sigs = sub["significant"].fillna(False).values
    bars = ax.bar(x + (i - 0.5) * width, vals, width, label=mode,
                  color=palette[mode], edgecolor="black", linewidth=0.5)
    for bar, sig in zip(bars, sigs):
        if sig:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + (0.001 if bar.get_height() >= 0 else -0.003),
                    "*", ha="center", va="bottom", fontsize=14, color="red")

# Annotate correlation sign on x-axis labels
r_vals = {
    row["ir_target"]: row["r_value"]
    for _, row in best_sae[["ir_target", "r_value"]].drop_duplicates().iterrows()
}
xlabels = [
    f"{t}\n(r={r_vals.get(t, 0):+.2f})" if r_vals.get(t) is not None else t
    for t in ir_targets
]

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel("ΔnDCG@10 (best α)")
ax.set_title(f"SAE Feature Steering by IR Target — Layer 17 ({DATASET})", fontweight="bold")
ax.legend(title="Mode")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "intervention_sae_effects.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Collapse Detection

In [ ]:
# Any score collapse?
probe_collapsed = probe_df[probe_df["collapsed"]]
sae_collapsed = sae_df[sae_df["collapsed"]]

print(f"Probe conditions with score collapse: {len(probe_collapsed)} / {len(probe_df)}")
if len(probe_collapsed) > 0:
    print(probe_collapsed[["target", "layer", "alpha_multiplier", "delta_ndcg"]].to_string())

print(f"\nSAE conditions with score collapse: {len(sae_collapsed)} / {len(sae_df)}")
if len(sae_collapsed) > 0:
    print(sae_collapsed[["feature_idx", "mode", "alpha", "delta_ndcg"]].to_string())

## 7. Full Summary Table

In [ ]:
# Significant results only
sig_probe = probe_df[probe_df["significant"]].sort_values("delta_ndcg", ascending=False, key=abs)
sig_sae = sae_df[sae_df["significant"]].sort_values("delta_ndcg", ascending=False, key=abs)

print(f"=== Significant Probe Steering ({len(sig_probe)}) ===")
if len(sig_probe) > 0:
    print(sig_probe[["target", "layer", "alpha_multiplier", "delta_ndcg", "delta_mrr", "p_value"]].to_string(index=False))
else:
    print("  None (p ≥ 0.05 for all conditions)")

print(f"\n=== Significant SAE Steering ({len(sig_sae)}) ===")
if len(sig_sae) > 0:
    print(sig_sae[["ir_target", "feature_idx", "r_value", "mode", "alpha",
                    "delta_ndcg", "delta_mrr", "p_value"]].to_string(index=False))
else:
    print("  None (p ≥ 0.05 for all conditions)")

## 8. Narrative Interpretation

Fill this in after running the experiments.

In [ ]:
# Print interpretation template
n_sig_probe = len(probe_df[probe_df["significant"]])
n_sig_sae = len(sae_df[sae_df["significant"]])
best_probe = probe_df.loc[probe_df["delta_ndcg"].abs().idxmax()]

print("=" * 70)
print("PHASE 8 SUMMARY")
print("=" * 70)
print(f"Baseline nDCG@10: {baseline_metrics['ndcg@10']:.4f}")
print(f"")
print(f"Probe steering: {n_sig_probe}/{len(probe_df)} conditions significant (p<0.05)")
print(f"Best probe effect: target={best_probe['target']}, layer={best_probe['layer']}, "
      f"α={best_probe['alpha_multiplier']:+.0f}, ΔnDCG={best_probe['delta_ndcg']:+.4f}")
print(f"")
print(f"SAE steering: {n_sig_sae}/{len(sae_df)} conditions significant")
print(f"")
print("Key question — probe faithfulness:")
print("  Do features with higher probe AUROC/R² produce larger causal ΔnDCG? (see cell 4)")